For each NACE Class get the 100 chunks that scored highest across all the reports 

In [157]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt
os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
from test_base import *

In [158]:
overview_path = "data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"

In [159]:
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "results/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"

In [160]:
reports = glob.glob(raw_data_path + "*/*_short.csv")
reports = glob.glob(raw_data_path + "*/*_long.csv")
len(reports)

1556

In [161]:
#sample_ratio = 1

In [162]:
#max_elements_per_class = 1000000
#top_k_sentences = 200000

In [163]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
#filter_only_right_chunks = True

In [164]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
#with_null_classifiers = False

In [165]:
new_threshold_cos_sin = 0.4

In [166]:
nace_level_descriptions = 2
nace_level = 1
assert nace_level_descriptions >= nace_level

In [167]:
training_data_path = "data/training_data/"

In [168]:
#suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"
suffix = f"2nd_approach" + f"__nace_level_{nace_level}__cos_thres_{new_threshold_cos_sin}"

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path
os.makedirs(end_path, exist_ok=True)

In [169]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path)
df_overview.head()

,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
1,35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
2,80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
3,49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
4,73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [170]:
df_nace_codes_descriptions = pd.read_csv("data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

For each class c: those paragraphs p of reports in class c with cos-sim(p, c) > 0.5

In [171]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    df = pd.read_csv(report)
    
    report_name = os.path.basename(report).replace(".txt_long.csv", "") + ".pdf"
    report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
    report_code = get_all_level(report_code)[nace_level]

    scores = [c for c in df.columns if "Scores" in c]
    
    df["max_class_sim"] = [scores[i][7] for i in np.argmax(df[scores], 1)]
    df["Score"] = df[scores].max(1)
    
    df.loc[(df["max_class_sim"] == report_code) & (df["Score"] > new_threshold_cos_sin),"NACE_Code"] = report_code
    df["NACE_Code"] = df["NACE_Code"].fillna("NO_CLASS")

    result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])


  0%|                                                                            | 0/1556 [00:00<?, ?it/s]/tmp/ipykernel_3583095/640071683.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, df[["Sentences", "Score", "NACE_Code"]]])
100%|█████████████████████████████████████████████████████████████████| 1556/1556 [01:04<00:00, 24.29it/s]


In [172]:
result.groupby("NACE_Code").count()

,Sentences,Score
NACE_Code,,
A,998,998
B,6023,6023
C,973,973
D,8768,8768
E,3001,3001
F,6837,6837
G,3869,3869
H,5931,5931
I,1850,1850


In [173]:
result_right = result[result["NACE_Code"] != "NO_CLASS"]
result_NO_CLASS = result.loc[result["NACE_Code"] == "NO_CLASS"].sample(n=20000)
result_final = pd.concat([result_right, result_NO_CLASS], axis=0)

In [174]:
result_final.groupby("NACE_Code").agg({
    "Score": "mean", 
    "Sentences": "count"
})

,Score,Sentences
NACE_Code,,
A,0.440772,998
B,0.461701,6023
C,0.452915,973
D,0.457924,8768
E,0.477387,3001
F,0.484579,6837
G,0.446147,3869
H,0.459307,5931
I,0.451663,1850


In [175]:
# recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

# df_recordings = pd.DataFrame(recordings)
# df_recordings = df_recordings.sort_values(by="Code")
# df_recordings.head()

#df_recordings.to_csv(end_path + "/statistics.csv")

In [176]:
full_df= result_final.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,text,Score,NACE_Code
19,as part of the naturaleaf asset acquistion the...,0.445046,A
130,for the year ended december and investing acti...,0.432593,A
22,management has determined that the consolidate...,0.412389,D
305,revenue from the sale of electricity and envir...,0.468761,D
339,this relates to revenue earned by ks from sale...,0.445356,D
...,...,...,...
489,the groups internal audit function follows up ...,0.249462,NO_CLASS
313,at december no deferred tax has been recognise...,0.328938,NO_CLASS
360,the management reports the findings to the boa...,0.293752,NO_CLASS
441,stage stage stage stage debt securities minimu...,0.396857,NO_CLASS


In [177]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 54304, Test size: 18101, Validation size: 18102


In [178]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [179]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [180]:
end_path

'data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.4'